# 🧠 Modelo D — Análise de sobrevivência (Cox PH)

**Pergunta:** quanto tempo até o caso fechar em acordo?

Eventos: `Acordo Firmado` → evento = 1. Demais → censurado. Tempo: dias entre `data_envio` e último pagamento.

Usamos `lifelines` com Cox Proportional Hazards.

In [1]:
import sys
from pathlib import Path
PROJECT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT))

import pandas as pd, numpy as np, joblib
GOLD   = PROJECT / 'notebooks' / 'data' / 'gold'
SILVER = PROJECT / 'notebooks' / 'data' / 'silver'
MODELS = PROJECT / 'models'

feat = pd.read_parquet(GOLD / 'contract_features.parquet')
contracts = pd.read_parquet(SILVER / 'contracts.parquet')[['id_contrato', 'data_envio']]

df = feat.merge(contracts, on='id_contrato')
df.shape

(10000, 20)

## 1. Construir tempo até evento

`tempo` = dias entre `data_envio` e `ultimo_pagamento` (ou `now()` se ainda não fechou).
`evento` = 1 se status = `Acordo Firmado`, 0 caso contrário.

In [2]:
ref = pd.Timestamp.now()
df['fim'] = df['ultimo_pagamento'].fillna(ref)
df['tempo_dias'] = (df['fim'] - df['data_envio']).dt.days.clip(lower=1)
df['evento'] = (df['status_cobranca'] == 'Acordo Firmado').astype(int)

df[['tempo_dias', 'evento']].describe()

,tempo_dias,evento
count,10000.000000,10000.000000
mean,20.574600,0.340700
std,32.132549,0.473968
min,1.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,0.000000
75%,33.000000,1.000000
max,197.000000,1.000000


## 2. Preparar dataset para Cox

Cox PH não aceita categóricas diretas — vamos `get_dummies`.

In [3]:
cols_num = ['score_risco', 'dias_atraso_inicial', 'valor_inadimplente',
            'taxa_adimplencia']
cols_cat = ['regiao', 'nome_assessoria', 'faixa_valor']

X = df[cols_num + cols_cat + ['tempo_dias', 'evento']].copy()
X = X.dropna()
X = pd.get_dummies(X, columns=cols_cat, drop_first=True)
X.shape

(9950, 15)

In [4]:
from lifelines import CoxPHFitter

cph = CoxPHFitter(penalizer=0.1)
cph.fit(X, duration_col='tempo_dias', event_col='evento')
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 9950 total observations, 6558 right-censored observations>
             duration col = 'tempo_dias'
                event col = 'evento'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 9950
number of events observed = 3392
   partial log-likelihood = -28346.21
         time fit was run = 2026-05-30 17:25:20 UTC

---
                                              coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                     
score_risco                                   0.00      1.00      0.00           -0.00            0.00                1.00                1.00
dias_atraso_inicial                          -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
valor_inadimplente                           -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
taxa_adimplencia                              0.15      1.16      0.10           -0.05            0.34                0.95                1.40
regiao_Nordeste                               0.00      1.00      0.04           -0.07            0.08                0.93                1.08
regiao_Norte                                 -0.02      0.98      0.08           -0.17            0.12                0.84                1.13
regiao_Sudeste                               -0.01      0.99      0.04           -0.08            0.07                0.92                1.08
regiao_Sul                                    0.06      1.06      0.05           -0.04            0.15                0.96                1.16
nome_assessoria_Fênix Recuperação De Crédito -0.02      0.99      0.04           -0.09            0.06                0.92                1.06
nome_assessoria_Nexus Mediação Financeira    -0.00      1.00      0.04           -0.08            0.08                0.92                1.08
nome_assessoria_Vértice Asset & Cobrança      0.01      1.01      0.04           -0.07            0.08                0.94                1.08
faixa_valor_baixo                            -0.01      0.99      0.05           -0.11            0.08                0.90                1.09
faixa_valor_medio                             0.03      1.03      0.05           -0.06            0.12                0.94                1.13

                                              cmp to     z    p  -log2(p)
covariate                                                                
score_risco                                     0.00  0.40 0.69      0.54
dias_atraso_inicial                             0.00 -0.50 0.62      0.70
valor_inadimplente                              0.00 -0.66 0.51      0.98
taxa_adimplencia                                0.00  1.47 0.14      2.82
regiao_Nordeste                                 0.00  0.09 0.93      0.10
regiao_Norte                                    0.00 -0.31 0.76      0.40
regiao_Sudeste                                  0.00 -0.15 0.88      0.19
regiao_Sul                                      0.00  1.19 0.23      2.09
nome_assessoria_Fênix Recuperação De Crédito    0.00 -0.40 0.69      0.54
nome_assessoria_Nexus Mediação Financeira       0.00 -0.05 0.96      0.05
nome_assessoria_Vértice Asset & Cobrança        0.00  0.19 0.85      0.23
faixa_valor_baixo                               0.00 -0.22 0.83      0.27
faixa_valor_medio                               0.00  0.66 0.51      0.97
---
Concordance = 0.52
Partial AIC = 56718.42
log-likelihood ratio test = 6.27 on 13 df
-log2(p) of ll-ratio test = 0.10

In [5]:
print(f'Concordance index : {cph.concordance_index_:.4f}')
print(f'Log-likelihood    : {cph.log_likelihood_:.2f}')

Concordance index : 0.5210
Log-likelihood    : -28346.21


In [6]:
# Persistência
artifact = {
    'model': cph,
    'feature_columns': [c for c in X.columns if c not in ('tempo_dias', 'evento')],
    'version': '1.0.0',
    'metrics': {'concordance_index': float(cph.concordance_index_)},
}
out = MODELS / 'survival_v1.joblib'
joblib.dump(artifact, out)
print('Salvo em', out)

Salvo em /home/ivissonalves/Workspaces/previsia/previsia-api/models/survival_v1.joblib
